# Introduction



## Problem Statement

Although extensive public data exist on poverty, educational attainment, unemployment, and population, these indicators are frequently stored in siloed systems and analyzed independently. This fragmentation limits the ability of public institutions—particularly those in rural regions—to conduct integrated, county-level assessments of regional need. As a result, many communities lack the analytic capacity to derive actionable insights that could inform local policy, educational programming, or economic development strategies.

The core data challenge is *twofold*:

1. Pattern Identification: Can we identify meaningful patterns and disparities in educational and economic conditions across counties in North Central Arkansas?

2. Predictive Insight: Can machine learning models be used to forecast future outcomes or classify counties based on shared vulnerabilities or developmental potential?

Addressing this challenge requires aggregating and standardizing multiple publicly available datasets, performing exploratory data analysis (EDA) to uncover relationships, and applying basic predictive modeling techniques to generate deeper insights. These insights can support more data-informed curriculum design, targeted workforce development programs, and competitive grant applications aligned with regional priorities.

## Section 1:  Work environment set up

In [ ]:
# --- Standard Imports ---
import pandas as pd
import numpy as np
import warnings
import logging
import matplotlib.pyplot as plt
import seaborn as sns
import re
import sys



from pathlib import Path
from functools import reduce
from sklearn.preprocessing import MinMaxScaler
from collections import Counter

# --- Setup ---
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
warnings.filterwarnings('ignore')

# --- Paths ---
data_dir = Path("data")
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
figures_dir = output_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)

image_dir = Path("outputs/images")
image_dir.mkdir(parents=True, exist_ok=True)

# Clear existing handlers
logging.getLogger().handlers.clear()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_dir / "project.log", mode='w', encoding='utf-8'),
        logging.StreamHandler(sys.stdout)
    ]
)


print("Environment ready. Paths and logging configured.")



# Section 2: Load and prepare Data sets

## Section 2.1: Load utility functions and national datasets ----

In [ ]:
# Section 2A: Load utility functions and national datasets ----

# Import custom utility functions
from utils import (
    standardize_column_names,
    extract_key_indicators,
    filter_to_county_level,
    log_duplicate_attributes,
    clean_and_extract_year,
    nca_counties
)

data_dir = Path("data")
complete_dir = data_dir / "complete_sets"

complete_files = {
    "edu": "Education2023.csv",
    "pop": "PopulationEstimates.csv",
    "poverty": "Poverty2023.csv",
    "unemp": "Unemployment2023.csv"
}


# Load, clean, and extract year from each dataset
complete_data = {}

for key, filename in complete_files.items():
    path = complete_dir / filename
    try:
        df = pd.read_csv(path, encoding='cp1252')
        df = standardize_column_names(df)
        logging.info(f" {key} columns after cleaning: {df.columns.tolist()}")
        df = filter_to_county_level(df)
        df = clean_and_extract_year(df)

        # Rename fields if necessary
        df.rename(columns={'area_name': 'county'}, inplace=True)

        log_duplicate_attributes(df, key)
        complete_data[key] = df
        logging.info(f"Loaded {filename}: {df.shape[0]} rows")

    except Exception as e:
        logging.error(f" Failed to load {filename}: {e}")


# Confirmation message
logging.info("Utility functions from utils.py loaded successfully.")


## Section 2.2: Load and process national datasets

In [ ]:
# Section 2.2: Load and process national datasets

complete_dir = data_dir / "complete_sets"
complete_files = {
    "edu": "Education2023.csv",
    "pop": "PopulationEstimates.csv",
    "poverty": "Poverty2023.csv",
    "unemp": "Unemployment2023.csv"
}

state_lookup = None

# Choose your analysis year here
selected_year = '2023'

for key, filename in complete_files.items():
    path = complete_dir / filename
    try:
        # Load and standardize
        df = pd.read_csv(path, encoding='cp1252')
        df = standardize_column_names(df)
        df = filter_to_county_level(df)
        df.rename(columns={'area_name': 'county', 'fips_code': 'fips', 'fipstxt': 'fips'}, inplace=True)
        df['county'] = df['county'].str.strip().str.lower()

        # Extract year from attribute
        if 'attribute' in df.columns:
            df = clean_and_extract_year(df)

            # Only filter datasets where year tagging applies
            if key in ['pop', 'poverty', 'unemp']:
                df = df[df['attribute_year'] == selected_year]

        # Save state info once from education
        if key == 'edu':
            state_lookup = df[['county', 'state']].drop_duplicates().copy()
            state_lookup['county'] = state_lookup['county'].str.lower().str.strip()
            state_lookup['state'] = state_lookup['state'].str.upper().str.strip()

        log_duplicate_attributes(df, key)
        complete_data[key] = df
        logging.info(f" Loaded and processed {filename} with year filter: {selected_year if key != 'edu' else 'N/A'}")

    except Exception as e:
        logging.error(f" Failed to load {filename}: {e}")


### Section 2.3: Discover Common Year Across Datasets

In [ ]:
# Section 2.3: Discover Common Year Across Datasets

def get_attribute_year_counts(df):
    """Counts how often each extracted attribute_year appears."""
    if 'attribute' in df.columns:
        df = clean_and_extract_year(df)
        return Counter(df['attribute_year'].dropna().astype(str))
    return {}

year_summary = {}

# Loop through each dataset and extract year counts
for key, df in complete_data.items():
    year_counts = get_attribute_year_counts(df)
    year_summary[key] = year_counts

# Display results
print("Year coverage by dataset:")
all_years = set()

for key, counts in year_summary.items():
    print(f"\n {key.upper()}:")
    if counts:
        for year, count in sorted(counts.items()):
            print(f"  {year}: {count}")
            all_years.add(year)
    else:
        print("  No attribute_year values found.")

# Find common years across all datasets that have year values
datasets_with_years = [set(c.keys()) for c in year_summary.values() if c]
common_years = set.intersection(*datasets_with_years) if datasets_with_years else set()



print("\n Common years across all datasets with usable attribute_year:", sorted(common_years))


## Section 2.4: Subset Arkansas and NCA Counties (No Full Merge)

In [ ]:
# Section 2.4: Subset Arkansas and NCA Counties (No Full Merge)

# Define NCA counties (lowercase, cleaned)
nca_cleaned = [c.lower().strip() for c in nca_counties]

# Function to filter, clean, and subset each dataset
def get_nca_subset(df, dataset_name):
    df = df.copy()
    df = df[df['state'].str.upper() == 'AR']
    df['county'] = (
        df['county']
        .str.replace(" county", "", regex=False)
        .str.replace(", ar", "", regex=False)
        .str.strip()
        .str.lower()
    )
    df_nca = df[df['county'].isin(nca_cleaned)].copy()
    logging.info(f"{dataset_name.upper()} → NCA rows: {df_nca.shape[0]}")
    return df_nca

# Get cleaned NCA subsets
edu_nca = get_nca_subset(complete_data['edu'], 'edu')
poverty_nca = get_nca_subset(complete_data['poverty'], 'poverty')
unemp_nca = get_nca_subset(complete_data['unemp'], 'unemp')
pop_nca = get_nca_subset(complete_data['pop'], 'pop')

# Pivot each to wide format
edu_wide = edu_nca.pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
poverty_wide = poverty_nca.pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
unemp_wide = unemp_nca.pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
pop_wide = pop_nca.pivot_table(index='county', columns='attribute', values='value', aggfunc='first')

# Merge NCA-wide datasets
df_nca = reduce(
    lambda left, right: pd.merge(left, right, on='county', how='outer'),
    [edu_wide, poverty_wide, unemp_wide, pop_wide]
)
# Flatten MultiIndex columns (from pivot operations)
df_nca.columns = [col if isinstance(col, str) else col[1] for col in df_nca.columns]

# Save and log
df_nca.to_csv(output_dir / f"nca_dataset_cleaned_{selected_year}.csv")
logging.info(f"NCA final dataset saved: nca_dataset_cleaned_{selected_year}.csv")
print(f"NCA dataset ready with shape: {df_nca.shape}")

## Section 2.5 Streamline NCA data set 

In [ ]:
# Section 2.5 Streamline NCA data set 

print("Columns BEFORE renaming:")
print(df_nca.columns.tolist())

# Ensure 'county' is a column, not just an index
df_nca = df_nca.reset_index()

# Rename long education column labels for clarity
df_nca = df_nca.rename(columns={
    "percent of adults who are high school graduates (or equivalent), 2019-23": "HighSchoolGradRate",
    "percent of adults with a bachelor's degree or higher, 2019-23": "BachelorsDegreeRate",
    "pctpovall_2023": "PovertyRate",
    "unemployment_rate_2023": "UnemploymentRate",
    "pop_estimate_2023": "Population"
})


# If the education rates are actually raw counts (verify first!)
df_nca['BachelorsDegreePct'] = (df_nca['BachelorsDegreeRate'] / df_nca['Population']) * 100
df_nca['HighSchoolGradPct'] = (df_nca['HighSchoolGradRate'] / df_nca['Population']) * 100


summary_vars = [
    'BachelorsDegreePct',
    'HighSchoolGradPct',
    'PovertyRate',
    'UnemploymentRate',
    'Population'
]

# Select only the summary variables and 'county'
summary_vars = [var for var in summary_vars if var in df_nca.columns]

# Log final columns
logging.info(f"Final NCA columns: {df_nca.columns.tolist()}")

df_nca_final = df_nca[summary_vars + ['county']].set_index('county')

df_nca_final.to_csv(output_dir / "nca_final.csv", index=True)
logging.info("✅ Final analysis dataset saved as: nca_final.csv")



# Section 3 Exploratory Data Overview (Tables & Summary Insights)

## Section 3.1 Dataset Overview

In [ ]:
# Section 3.1: Dataset Overview
print("Dataset Dimensions (rows, columns):", df_nca_final.shape)

print("\n Column Names:")
for col in df_nca_final.columns:
    print(" -", col)

print("\n Dataset Info:")
df_nca_final.info()

print("\n Preview of First 5 Rows:")
display(df_nca_final.head())

## Section 3.2 Data Dictionary 

In [ ]:
# Section 3.2: Data Dictionary Summary (for df_nca_final)

# Manually define the variable descriptions (aligned with df_nca_final)
data_dictionary = {
    "BachelorsDegreePct": "Percent of population with a bachelor’s degree (calculated from rate and population)",
    "HighSchoolGradPct": "Percent of population with a high school diploma or equivalent (calculated)",
    "PovertyRate": "Estimated percentage of residents living below the poverty line (2023)",
    "UnemploymentRate": "Estimated unemployment rate of labor force (2023)",
    "Population": "Estimated total population of the county (2023)"
}

# Convert to a DataFrame for display and export
dictionary_df = pd.DataFrame.from_dict(data_dictionary, orient='index', columns=['Description'])
dictionary_df.index.name = 'Variable'

# Display and save
display(dictionary_df)

dictionary_df.to_csv(output_dir / "nca_data_dictionary.csv")
logging.info("📖 Data dictionary saved to: nca_data_dictionary.csv")


## Section 3.3 Descriptive Statistics

In [ ]:
# Section 3.3: Per-Variable County-Level Tables

percent_vars = [
    'BachelorsDegreePct',
    'HighSchoolGradPct',
    'PovertyRate',
    'UnemploymentRate'
]

for var in percent_vars:
    var_table = df_nca_final[[var]].copy()
    var_table.columns = ['Value']
    var_table.index.name = 'County'

     # Round values
    var_table = var_table.round(3)

    # Display
    print(f"\n📊 {var} by County:")
    display(var_table)

        # Save to CSV
    filename = f"section3_2_{var.lower()}_by_county.csv"
    var_table.to_csv(output_dir / filename)
    logging.info(f"[Section 3.2] Saved: {filename}")


In [ ]:
# Auto-Summary Per Variable

for var in percent_vars:
    series = df_nca_final[var]
    max_val = series.max()
    min_val = series.min()
    mean_val = series.mean()
    std_val = series.std()

    max_county = series.idxmax()
    min_county = series.idxmin()

    print(f"\n🧠 Insight Summary for {var}:")
    print(f"  - Average across counties: {mean_val:.3f}")
    print(f"  - Standard deviation: {std_val:.3f}")
    print(f"  - Highest: {max_val:.3f} ({max_county.title()})")
    print(f"  - Lowest: {min_val:.3f} ({min_county.title()})")


## Section 3.4 Outlier/Range Table

In [ ]:
# Section 3.5: Outlier / Range Table

print(" Outlier and Range Summary for Selected Indicators:\n")


for var in summary_vars:
    print(f" Top 3 counties by {var}:")
    display(df_nca_final[[var]].sort_values(by=var, ascending=False).head(3))
    
    print(f" Bottom 3 counties by {var}:")
    display(df_nca_final[[var]].sort_values(by=var, ascending=True).head(3))
    
    print("-" * 60)


## Section 3.5 County Summary Table

In [ ]:
# Section 3.4: One CSV per County Profile

export_vars = percent_vars + ['Population']

for county, row in df_nca_final[export_vars].iterrows():
    county_df = row.to_frame(name='Value')
    county_df.index.name = 'Variable'
    
    filename = f"section3_4_{county.lower().replace(' ', '_')}_profile.csv"
    county_df.to_csv(output_dir / filename)
    logging.info(f"Saved profile: {filename}")


# Section 4 Data Visualizations & Pattern Discovery

## Section 4.1: Extract and Name Key Indicator Variables

In [ ]:
# Section 4.1: Extract and Name Key Indicator Variables

# Define key indicators to plot
variables = [
    'BachelorsDegreePct',
    'HighSchoolGradPct',
    'PovertyRate',
    'UnemploymentRate',
    'Population'
]

print("🎯 Variables selected for visualization:")
for var in variables:
    print(f" - {var}")

print("\n📋 Preview of data to visualize:")
display(df_nca_final[variables].head())

# Optional: Save preview for report
df_nca_final[variables].head().to_csv(figures_dir / "section4_2_preview_visualization_data.csv")
logging.info("Saved visualization data preview for Section 4.2.")


## Section 4.2 Visualization 

### Section 4.2.1: Distribution Plots (Histogram + KDE)

In [ ]:
# Section 4.2.1: Distribution Plots (Histogram + KDE)

# Variables to plot
variables = [
    'BachelorsDegreePct',
    'HighSchoolGradPct',
    'PovertyRate',
    'UnemploymentRate'
]

for var in variables:
    plt.figure(figsize=(8, 4))
    sns.histplot(df_nca_final[var], kde=True, bins=10, color='skyblue')

    plt.title(f"Distribution of {var}", fontsize=14)
    plt.xlabel(f"{var} (%)", fontsize=12)
    plt.ylabel("Number of Counties", fontsize=12)
    plt.tight_layout()

    # Save
    filename = f"section4_2_1_distribution_{var.lower()}.png"
    plt.savefig(image_dir / filename, dpi=300, bbox_inches='tight')
    logging.info(f"📊 Saved: {filename}")
    plt.show()


### Section 4.2.2: Dot Plot of BachelorsDegreePct by County

In [ ]:
# Section 4.2.2: Dot Plot of BachelorsDegreePct by County


# Reset index so 'county' becomes a column
df_plot = df_nca_final.reset_index()

# Sort for cleaner plot
df_plot = df_plot.sort_values('BachelorsDegreePct')

plt.figure(figsize=(10, 6))
sns.stripplot(y='county', x='BachelorsDegreePct', data=df_plot, size=8, color='steelblue', jitter=False)

plt.title("Percent of Adults with Bachelor's Degree by County", fontsize=14)
plt.xlabel("BachelorsDegreePct (%)")
plt.ylabel("County")
plt.tight_layout()

# Save
filename = "section4_2_2_dotplot_bachelorsdegreepct.png"
plt.savefig(image_dir / filename, dpi=300, bbox_inches='tight')
logging.info(f"📍 Saved: {filename}")
plt.show()
